<a href="https://colab.research.google.com/github/harshvarudkar/test/blob/master/Demo0612.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Without memory

In [1]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [2]:
!pip install -qU langchain langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.2/552.2 kB 10.8 MB/s eta 0:00:00


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers short."),
    ("human", "{input}")
])

In [7]:
chain = prompt | llm

In [8]:
print("---- Turn 1 ----")
response1 = chain.invoke({"input": "Hi, my name is Harsh Varudkar."})
print("User: Hi, my name is Harsh Varudkar.")
print("AI:", response1.content)

---- Turn 1 ----
User: Hi, my name is Harsh Varudkar.
AI: Hi Harsh! How can I assist you today?


In [9]:
print("\n---- Turn 2 ----")
response2 = chain.invoke({"input": "What is my name?"})
print("User: What is my name?")
print("AI:", response2.content)


---- Turn 2 ----
User: What is my name?
AI: I don't know your name. You haven't provided it.


In [10]:
print("\nExpected issue:")
print("- The second call may not know your name.")
print("- Reason: no chat history is passed to the model.")


Expected issue:
- The second call may not know your name.
- Reason: no chat history is passed to the model.


**What issue it demonstrates**
The second prompt is handled as a brand-new request, so the model may fail to answer “What is my name?” correctly because no prior context was included. That is the normal behavior of a stateless setup.

With Memory

In [11]:
!pip install -qU langchain langchain-openai langchain-core

In [12]:
from collections import defaultdict

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [13]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep answers short."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

In [15]:
chain = prompt | llm

In [16]:
store = defaultdict(InMemoryChatMessageHistory)

In [17]:
def get_session_history(session_id: str):
    return store[session_id]


In [18]:
chat_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [19]:
config = {"configurable": {"session_id": "demo-user"}}

In [27]:
print("---- Turn 1 ----")
response1 = chat_with_memory.invoke(
    {"input": "Hi, my name is Harsh Varudkar1."},
    config=config
)
print("User: Hi, my name is Harsh Varudkar1.")
print("AI:", response1.content)

---- Turn 1 ----
User: Hi, my name is Harsh Varudkar1.
AI: Hi Harsh Varudkar1! How can I assist you today?


In [28]:
print("\n---- Turn 2 ----")
response2 = chat_with_memory.invoke(
    {"input": "What is my name?"},
    config=config
)
print("User: What is my name?")
print("AI:", response2.content)


---- Turn 2 ----
User: What is my name?
AI: Your name is Harsh Varudkar1.


In [29]:
print("\n---- Stored history ----")
for msg in store["demo-user"].messages:
    print(type(msg).__name__, ":", msg.content)


---- Stored history ----
HumanMessage : Hi, my name is Alice.
AIMessage : Hi Alice! How can I assist you today?
HumanMessage : Hi, my name is Alice.
AIMessage : Hello again, Alice! How can I help you today?
HumanMessage : What is my name?
AIMessage : Your name is Alice.
HumanMessage : Hi, my name is Harsh Varudkar.
AIMessage : Hi Harsh Varudkar! How can I assist you today?
HumanMessage : What is my name?
AIMessage : Your name is Harsh Varudkar.
HumanMessage : Hi, my name is Harsh Varudkar1.
AIMessage : Hi Harsh Varudkar1! How can I assist you today?
HumanMessage : What is my name?
AIMessage : Your name is Harsh Varudkar1.
